### Prerequisites:

Before we dive into the implementation, we have to install some prerequisites:

 - we use [pandas](https://pandas.pydata.org/docs/) to easily load and format our data
 - we use Qwen's open-source LLM [Qwen 2.5-7B](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct) for zero-shot classification

In [1]:
%%capture
# install dependencies:
!pip install numpy matplotlib seaborn torch pandas scikit-learn transformers accelerate tqdm datasets

### Data processing

In [2]:
from datasets import load_dataset

train_data = load_dataset("go_emotions", "simplified", split="train")
test_data  = load_dataset("go_emotions", "simplified", split="test")

print(f"Train set size: {len(train_data)} examples")
print(f"Test set size:  {len(test_data)} examples")

Train set size: 43410 examples
Test set size:  5427 examples


In [3]:
# filter to keep only single-label examples:
def is_single_label(example):
    return len(example["labels"]) == 1

train_data = train_data.filter(is_single_label)
test_data  = test_data.filter(is_single_label)

# flatten the label from list to scalar:
def flatten_label(example):
    example["label"] = example["labels"][0]
    return example

train_data = train_data.map(flatten_label)
test_data  = test_data.map(flatten_label)

In [4]:
# get the label names:
label_names = train_data.features["labels"].feature.names

# create a mapping from index to label:
index2label = {i: name for i, name in enumerate(label_names)}

In [5]:
# convert datasets to dataframes:
train_df = train_data.to_pandas()
test_df  = test_data.to_pandas()

train_df.sample()

,text,labels,id,label
31495,Pretty sure it’s going on in the apartment acr...,[3],eeeqm4x,3


In [6]:
from sklearn.model_selection import train_test_split

# sample 2k training examples stratified by label:
train_df, _ = train_test_split(
    train_df, stratify=train_df["label"], train_size=2000, random_state=42
)

# sample 1k test examples stratified by label:
test_df, _ = train_test_split(
    test_df, stratify=test_df["label"], train_size=1000, random_state=42
)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

Train: 2000, Test: 1000


In [7]:
# map numeric labels to string labels:
train_df["label"] = train_df["label"].map(index2label)
test_df["label"]  = test_df["label"].map(index2label)

In [8]:
import numpy as np

# select input and label from data:
X_train = train_df["text"].to_numpy()
y_train = train_df["label"].to_numpy()

X_test  = test_df["text"].to_numpy()
y_test  = test_df["label"].to_numpy()

classes = sorted(set(y_train))
print("Classes:", len(classes))

Classes: 28


### Instantiate Qwen 2.5

In order to be able to use huggingface's Qwen 2.5-7B models, we first need to log in to huggingface (you can request access to the model [here](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)).

In [9]:
# import getpass
# from huggingface_hub import login
# login(getpass.getpass('Enter your huggingface API-key:'))

In this tutorial we will use the standard huggingface text-generation pipeline (compressed to 16 bit floating point weights) for the instruction tuned Qwen 2.5-7B model.

In [10]:
import transformers
import torch

# create llm pipeline:
llm = transformers.pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto"
)

# Get special tokens for later:
bos_token_id = llm.tokenizer.bos_token_id
eos_token_id = llm.tokenizer.eos_token_id
pad_token_id = llm.tokenizer.pad_token_id or llm.tokenizer.eos_token_id

`torch_dtype` is deprecated! Use `dtype` instead!


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Results Handling Helper

In [11]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import json

def save_results(y_true, y_pred, classes, filename="results.json"):
    report   = classification_report(y_true, y_pred, labels=classes, zero_division=0, output_dict=True)
    accuracy = accuracy_score(y_true, y_pred)
    cm       = confusion_matrix(y_true, y_pred, labels=classes)
    results  = {"classification_report": report, "accuracy": accuracy,
                "confusion_matrix": cm.tolist(), "labels": classes}
    with open(filename, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: '{filename}'")

In [12]:
import json
import numpy as np

def save_prompt_lengths(prompt_lengths, input_lengths, shots_lengths, num_shots, filename="prompt_lengths.json"):
    def stats(v): return {"mean": float(np.mean(v)), "min": int(np.min(v)),
                          "max": int(np.max(v)), "std": float(np.std(v)), "values": v}
    data = {"prompt_lengths": stats(prompt_lengths), "input_lengths": stats(input_lengths),
            "shots_lengths": stats(shots_lengths), "num_shots": stats(num_shots)}
    with open(filename, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Prompt length statistics saved to: '{filename}'")

### Zero-shot Classification

Build a prompt listing **all possible class labels** explicitly. No labelled examples are provided — the model must rely solely on the class names and task description.

In [ ]:
def get_zero_shot_prompt(text, classes):
    """Build a zero-shot classification prompt listing all possible classes."""
    replace_qm = lambda s: s.replace('"', "'")
    class_list = ", ".join(classes)
    prompt = (
        "We classify Reddit comments by their emotion. "
        f"The possible classes are: {class_list}.\n\n"
        "Please predict the correct class for the following sample. "
        "Output only the class label, nothing else."
    )
    return {"role": "user", "content": f'{prompt}\n\n\"{replace_qm(text)}\" => '}


# print a sample prompt to verify:
print(get_zero_shot_prompt(X_test[0], classes)["content"])

In [14]:
from tqdm import tqdm
import numpy as np

def zero_shot_batch(test, classes):
    predictions    = []
    prompt_lengths = []
    input_lengths  = []
    shots_lengths  = []  # always 0 — no shots
    num_shots      = []  # always 0

    for t in tqdm(test, desc="Zero-shot inference"):
        prompt = get_zero_shot_prompt(t, classes)

        prompt_lengths.append(len(llm.tokenizer(prompt["content"])["input_ids"]))
        input_lengths.append(len(llm.tokenizer(t)["input_ids"]))
        shots_lengths.append(0)
        num_shots.append(0)

        output = llm(
            [prompt],
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            max_new_tokens=5,
            do_sample=False,
            temperature=None,
            top_p=None
        )
        predictions.append(output[0]["generated_text"][-1]["content"].strip())

    return predictions, prompt_lengths, input_lengths, shots_lengths, num_shots


predictions, prompt_lengths, input_lengths, shots_lengths, num_shots = zero_shot_batch(X_test, classes)

Zero-shot inference:   0%|          | 0/1000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'bos_token_id', 'max_new_tokens', 'temperature', 'do_sample', 'top_p', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zero-shot inference:   0%|          | 1/1000 [00:00<13:00,  1.28it/s]

...

Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zero-shot inference: 100%|█████████▉| 999/1000 [04:49<00:00,  3.31it/s]

Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zero-shot inference: 100%|██████████| 1000/1000 [04:49<00:00,  3.49it/s]

Zero-shot inference: 100%|██████████| 1000/1000 [04:49<00:00,  3.46it/s]

In [15]:
predictions[:10]

["I'm sorry, but",
 "I'm sorry, but",
 "I'm sorry, but",
 "I'm sorry, but",
 "I'm sorry, but",
 "I'm sorry, but",
 "I'm sorry, but",
 "I'm sorry, but",
 "I'm sorry, but",
 "I'm sorry, but"]

In [16]:
from sklearn.metrics import classification_report, accuracy_score

print("Classification Report:\n", classification_report(y_test, predictions, labels=list(classes), zero_division=0))
print("Accuracy:", accuracy_score(y_test, predictions))

Classification Report:
                 precision    recall  f1-score   support

    admiration       0.00      0.00      0.00      76.0
     amusement       0.00      0.00      0.00      41.0
         anger       0.00      0.00      0.00      29.0
     annoyance       0.00      0.00      0.00      42.0
      approval       0.00      0.00      0.00      51.0
        caring       0.00      0.00      0.00      19.0
     confusion       0.00      0.00      0.00      21.0
     curiosity       0.00      0.00      0.00      38.0
        desire       0.00      0.00      0.00      12.0
disappointment       0.00      0.00      0.00      19.0
   disapproval       0.00      0.00      0.00      42.0
       disgust       0.00      0.00      0.00      17.0
 embarrassment       0.00      0.00      0.00       5.0
    excitement       0.00      0.00      0.00      12.0
          fear       0.00      0.00      0.00      14.0
     gratitude       0.00      0.00      0.00      57.0
         grief       0.

In [17]:
save_results(y_test, predictions, list(classes), filename="/home/v25/ippa6201/cicle-evaluation/go-emotions/results/predictions/go-emotions-qwen-2.5-7b-zeroshot-2.0k-samples.json")

Results saved to: '/home/v25/ippa6201/cicle-evaluation/go-emotions/results/predictions/go-emotions-qwen-2.5-7b-zeroshot-2.0k-samples.json'


In [18]:
save_prompt_lengths(prompt_lengths, input_lengths, shots_lengths, num_shots, filename="/home/v25/ippa6201/cicle-evaluation/go-emotions/results/lengths/go-emotions-qwen-2.5-7b-zeroshot-2.0k-samples.json")

Prompt length statistics saved to: '/home/v25/ippa6201/cicle-evaluation/go-emotions/results/lengths/go-emotions-qwen-2.5-7b-zeroshot-2.0k-samples.json'
